# DonkeyCar steering with ResNet50

Compare raw PID steering with training targets that interpolate isolated spikes. Run the cells in order after preparing the seeded indices as described in `doc/donkeycar/data.md`. The pretrained ResNet50 backbone stays frozen. All linear heads use identical initialization and minibatch order. Compare 1x, 2x and 4x loss weights on non-spike frames with absolute raw steering at least 0.3. All other frames keep unit weight. Select each checkpoint by MAE on original validation commands and score it on held-out road seeds.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from time import perf_counter
from uuid import uuid4

import numpy as np

from PIL import Image
from donkeycar_app.dataset import preprocess_camera
from mydezero import cuda, no_grad, test_mode
from mydezero.dataloaders import DataLoader
from mydezero.layers import Linear
from mydezero.models import ResNet50
from mydezero.optimizers import Adam

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the Project I repository.")

index_dir = root / "data/processed/donkeycar/seeded"
seed = 42
epochs = 200
image_batch_size = 32
feature_batch_size = 256
learning_rate = 3e-4
lr_patience = 5
lr_factor = 0.5
min_learning_rate = 1e-5
early_stop_patience = 15
min_mae_improvement = 5e-4
sharp_threshold = 0.3
sharp_multipliers = (1, 2, 4)
use_gpu = cuda.gpu_enable and cuda.cupy.cuda.runtime.getDeviceCount() > 0
np.random.seed(seed)
if use_gpu:
    cuda.cupy.random.seed(seed)
print("Device", "CUDA" if use_gpu else "CPU")

In [ ]:
indices = {split: index_dir / f"{split}.json" for split in ("train", "validation", "test")}
rows = {split: json.loads(path.read_text(encoding="utf-8")) for split, path in indices.items()}
seeds = {split: {r["seed"] for r in part} for split, part in rows.items()}
assert sum(map(len, seeds.values())) == len(set.union(*seeds.values()))
for split, part in rows.items():
    repaired = sum(r["raw"] != r["target"] for r in part)
    print(split, len(part), "frames,", repaired, "repaired labels")
    if split != "train":
        assert repaired == 0

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
run_dir = root / "runs" / "donkeycar" / run_id
weights_dir = root / "models" / "donkeycar" / run_id
run_dir.mkdir(parents=True)
weights_dir.mkdir(parents=True)
config = {
    "seeds": {split: sorted(values) for split, values in seeds.items()},
    "indices": {
        p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in indices.values()
    },
    "fit_samples": len(rows["train"]),
    "validation_samples": len(rows["validation"]),
    "model": "ResNet50 frozen pretrained backbone with a scalar linear head",
    "pretrained_weights": ResNet50.WEIGHTS_PATH.format(50),
    "image_preprocessing": "160x120, CHW float32, mean-centered BGR",
    "seed": seed,
    "epochs": epochs,
    "image_batch_size": image_batch_size,
    "feature_batch_size": feature_batch_size,
    "learning_rate": learning_rate,
    "lr_patience": lr_patience,
    "lr_factor": lr_factor,
    "min_learning_rate": min_learning_rate,
    "early_stop_patience": early_stop_patience,
    "min_mae_improvement": min_mae_improvement,
    "sharp_multipliers": sharp_multipliers,
    "sharp_threshold": sharp_threshold,
    "device": "cuda" if use_gpu else "cpu",
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print("Run directory", run_dir.relative_to(root))

In [ ]:
def extract_features(model, part, label):

    features = np.empty((len(part), 2048), dtype=np.float32)
    offset = 0
    started = perf_counter()
    next_report = 2048
    with test_mode(), no_grad():
        for start in range(0, len(part), image_batch_size):
            batch = []
            for row in part[start:start + image_batch_size]:
                with Image.open(root / row["image"]) as image:
                    batch.append(preprocess_camera(image))
            images = np.stack(batch)
            if use_gpu:
                images = cuda.as_cupy(images)
            end = offset + len(images)
            features[offset:end] = cuda.as_numpy(model.extract_features(images))
            offset = end
            if offset >= next_report or offset == len(part):
                print(f"{label} features {offset:,}/{len(part):,} in {perf_counter() - started:.1f}s")
                next_report += 2048
    if not np.isfinite(features).all():
        raise ValueError(f"Non-finite {label} features")
    return features


model = ResNet50(pretrained=True)
model.fc6 = Linear(1, in_size=2048)
if use_gpu:
    model.to_gpu()
fit_features = extract_features(model, rows["train"], "train")
validation_features = extract_features(model, rows["validation"], "validation")

def targets(part, key):
    return np.asarray([r[key] for r in part], dtype=np.float32).reshape(-1, 1)

validation_targets = targets(rows["validation"], "raw")

In [ ]:
def predict(head, features):
    with test_mode(), no_grad():
        values = head(cuda.as_cupy(features) if use_gpu else features)
    return np.clip(cuda.as_numpy(values), -1.0, 1.0)


def measure(prediction, target):
    if not np.isfinite(prediction).all():
        raise ValueError("Non-finite steering prediction")
    errors = np.abs(prediction - target)
    sharp = np.abs(target) >= sharp_threshold
    return {
        "samples": int(target.size),
        "mae": float(errors.mean()),
        "sharp_samples": int(sharp.sum()),
        "sharp_mae": float(errors[sharp].mean()) if sharp.any() else None,
        "zero_steering_mae": float(np.abs(target).mean()),
    }


sharp_train = np.array([abs(r["raw"]) >= sharp_threshold and not r["spike"]
                        for r in rows["train"]])


def train_head(recipe, fit_targets, multiplier):
    checkpoint = weights_dir / f"{recipe}.npz"
    np.random.seed(seed)
    model.fc6 = Linear(1, in_size=2048)
    if use_gpu:
        model.fc6.to_gpu()
    weights = np.where(sharp_train, multiplier, 1).astype(np.float32)
    labels = np.column_stack((fit_targets, weights))
    loader = DataLoader(list(zip(fit_features, labels)), feature_batch_size, gpu=use_gpu)
    optimizer = Adam(lr=learning_rate).setup(model.fc6)
    best_mae = float("inf")
    best_epoch = 0
    progress_mae = float("inf")
    epochs_without_progress = 0
    epochs_since_lr_change = 0
    stop_reason = "epoch_limit"
    with (run_dir / f"{recipe}-epochs.jsonl").open("w", encoding="utf-8") as metrics_file:
        for epoch in range(1, epochs + 1):
            weighted_loss = 0.0
            for features, batch_labels in loader:
                steering = batch_labels[:, :1]
                batch_weights = batch_labels[:, 1:]
                residual = model.fc6(features) - steering
                weight_sum = float(batch_weights.sum())
                loss = (residual ** 2 * batch_weights).sum() / weight_sum
                model.fc6.cleargrads()
                loss.backward()
                optimizer.update()
                weighted_loss += float(loss.data.item()) * weight_sum
            validation = measure(predict(model.fc6, validation_features), validation_targets)
            val_mae = validation["mae"]
            if val_mae < best_mae:
                best_mae, best_epoch = val_mae, epoch
                best_weights = model.fc6.W.data.copy()
                best_bias = model.fc6.b.data.copy()

            if val_mae <= progress_mae - min_mae_improvement:
                progress_mae = val_mae
                epochs_without_progress = 0
                epochs_since_lr_change = 0
            else:
                epochs_without_progress += 1
                epochs_since_lr_change += 1

            epoch_lr = optimizer.lr
            should_stop = epochs_without_progress >= early_stop_patience
            if (
                not should_stop
                and epoch < epochs
                and epochs_since_lr_change >= lr_patience
                and epoch_lr > min_learning_rate
            ):
                optimizer.lr = max(min_learning_rate, epoch_lr * lr_factor)
                epochs_since_lr_change = 0
                print(f"Reducing learning rate to {optimizer.lr:.2g} for epoch {epoch + 1}")

            metrics = {
                "epoch": epoch,
                "weighted_train_mse": weighted_loss / float(weights.sum()),
                "validation": validation,
                "learning_rate": epoch_lr,
                "next_learning_rate": optimizer.lr,
                "epochs_without_progress": epochs_without_progress,
            }
            metrics_file.write(json.dumps(metrics, allow_nan=False) + "\n")
            metrics_file.flush()
            print(f"Epoch {epoch:2d}  weighted train MSE {metrics['weighted_train_mse']:.4f}  val MAE {validation['mae']:.4f}")
            if should_stop:
                stop_reason = "early_stopping"
                break

    epochs_completed = epoch
    print(f"Training finished after {epochs_completed} epochs ({stop_reason})")

    model.fc6.W.data = best_weights
    model.fc6.b.data = best_bias
    temporary_checkpoint = checkpoint.with_suffix(".partial.npz")
    model.save_weights(temporary_checkpoint)
    temporary_checkpoint.replace(checkpoint)
    print(f"Saved epoch {best_epoch} with validation MAE {best_mae:.4f}")
    return dict(checkpoint=checkpoint.relative_to(root).as_posix(), best_epoch=best_epoch,
                epochs_completed=epochs_completed, stop_reason=stop_reason, validation_mae=best_mae)

results = {}
for recipe, key in (("raw", "raw"), ("smoothed", "target")):
    for multiplier in sharp_multipliers:
        name = f"{recipe}_x{multiplier}"
        print(name)
        results[name] = train_head(name, targets(rows["train"], key), multiplier)

In [ ]:
test_rows = rows["test"]
test_targets = targets(test_rows, "raw")
route_labels = np.array([row["route"] for row in test_rows])
sharp_nonspike = np.array([abs(r["raw"]) >= sharp_threshold and not r["spike"] for r in test_rows])
offline_model = ResNet50()
offline_model.fc6 = Linear(1)
test_features = None
for recipe, result in results.items():
    offline_model.load_weights(root / result["checkpoint"])
    if use_gpu:
        offline_model.to_gpu()
    if test_features is None:
        test_features = extract_features(offline_model, test_rows, "test")
    predictions = predict(offline_model.fc6, test_features)
    result["overall"] = measure(predictions, test_targets)
    result["sharp_nonspike"] = measure(predictions[sharp_nonspike], test_targets[sharp_nonspike])
    result["by_route"] = {
        route: measure(predictions[route_labels == route], test_targets[route_labels == route])
        for route in sorted(set(route_labels))
    }
(run_dir / "test.json").write_text(json.dumps(results, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(results, indent=2))

The test metrics compare predictions with the recorded PID steering. The large-steering subset uses `abs(steering) >= 0.3`. The zero-steering MAE is the baseline from always predicting zero. This experiment measures offline imitation accuracy. Simulator driving is evaluated separately.